# Real-time customer segmentation tutorial

Generated from the FeatureMesh docs tutorial. This variant targets the FeatureMesh demos Jupyter environment.


## Set up FeatureMesh

Load the Jupyter magic and create local `ServingClient` / `BatchClient` instances for the demos Jupyter environment. Run these cells once before the tutorial.


In [1]:
%load_ext featuremesh

In [2]:
from featuremesh import RegistryDeployment, ServingClient, ServingDeployment, set_default
from libs.helpers.serving_executors import make_serving_executors
from libs.helpers.utils import get_postgres_url, get_redis_url

set_default("registry", RegistryDeployment.LOCAL)
set_default("serving", ServingDeployment.EMBEDDED)

serving_client = ServingClient()
set_default("client", serving_client)

# Use env from the demos stack (public compose: postgres/redis service
# names + internal ports; private jupyter: host.docker.internal + published ports).
SERVING_EXECUTORS = make_serving_executors(
    postgres=get_postgres_url(),
    redis=get_redis_url(),
)
client = serving_client

This tutorial builds a promotion decision from values in Redis, updates those values live, and compiles the rule for repeated low-latency calls.

By the end, you will have followed the complete serving lifecycle: seed data, define a source-backed feature, refresh the serving process, evaluate it, observe a live data change, and create a prepared statement.

Complete the [FeatureMesh homepage walkthrough](https://featuremesh.com/docs/tutorials/homepage/featuremesh) first if source features or serving refreshes are new. These examples require the `serving` backend, Redis, and a serving client. Run them from top to bottom because later steps reuse the persisted model and mutated Redis data.


## Meet the three customers

Each customer is stored as a Redis hash with two precomputed values:

- `days_since_order` measures recency.
- `lifetime_value_cents` stores currency as an integer.

Seed the three hashes:


In [3]:
_result = SERVING_EXECUTORS["redis"].statement("""DEL tutorial:segment:1 tutorial:segment:2 tutorial:segment:3
HSET tutorial:segment:1 days_since_order 45 lifetime_value_cents 15000
HSET tutorial:segment:2 days_since_order 10 lifetime_value_cents 8000
HSET tutorial:segment:3 days_since_order 60 lifetime_value_cents 5000
""")
if _result.errors:
    raise RuntimeError(_result.errors)

The starting values are:

- Customer `1`: 45 days, 15000 cents
- Customer `2`: 10 days, 8000 cents
- Customer `3`: 60 days, 5000 cents

Now reset the tutorial feature namespace so the model starts clean:


In [4]:
%%featureql --client serving_client

DROP FEATURES IF EXISTS IN FM.TUTORIALS.REALTIME_SEGMENTATION UP TO LEVEL 9;

""


The promotion rule will return true when the last order was at least 30 days ago **and** lifetime value is at least $100 (`10000` cents).

## Define the serving model

The model follows the same path the serving query will execute:

1. Bind a typed `CUSTOMER_ID`.
2. Build the corresponding Redis key.
3. Read the two hash fields with `EXTERNAL_KV()`.
4. Cast the external values to `BIGINT`.
5. Evaluate the `SHOW_PROMO` rule.


In [5]:
%%featureql --client serving_client

CREATE OR REPLACE FEATURES IN FM.TUTORIALS.REALTIME_SEGMENTATION AS
PERSIST
    CUSTOMERS := ENTITY(),
    CUSTOMER_ID := INPUT(BIGINT#CUSTOMERS),
    REDIS_SOURCE := SOURCE_KV(
        'redis://host.docker.internal:26379'
        WITH (timeout='500ms')
    ),
    REDIS_KEY := 'tutorial:segment:' || CAST(KEY_TO_SCALAR(CUSTOMER_ID) AS VARCHAR),
    DAYS_SINCE_ORDER := CAST(
        EXTERNAL_KV(KEY REDIS_KEY FIELD 'days_since_order' FROM REDIS_SOURCE)
        AS BIGINT
    ),
    LIFETIME_VALUE_CENTS := CAST(
        EXTERNAL_KV(KEY REDIS_KEY FIELD 'lifetime_value_cents' FROM REDIS_SOURCE)
        AS BIGINT
    ),
    SHOW_PROMO := DAYS_SINCE_ORDER >= 30 AND LIFETIME_VALUE_CENTS >= 10000
;

,FEATURE_NAME,STATUS,MESSAGE
0,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMERS,CREATED,Feature created as not exists
1,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,CREATED,Feature created as not exists
2,FM.TUTORIALS.REALTIME_SEGMENTATION.REDIS_SOURCE,CREATED,Feature created as not exists
3,FM.TUTORIALS.REALTIME_SEGMENTATION.REDIS_KEY,CREATED,Feature created as not exists
4,FM.TUTORIALS.REALTIME_SEGMENTATION.DAYS_SINCE_...,CREATED,Feature created as not exists
5,FM.TUTORIALS.REALTIME_SEGMENTATION.LIFETIME_VA...,CREATED,Feature created as not exists
6,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO,CREATED,Feature created as not exists


Persisting this definition updates the registry, but the running serving process installs the new connection only at its next registry sync. Refresh the serving engine before evaluating the model:


In [6]:
%%featureql --client serving_client

REFRESH SERVING;

,FEATURE,KIND,STATUS,MESSAGE,SINCE
0,FM.TUTORIALS.ONLINE_FEDERATED.LAST_ORDER_PRICE_PS,PREPARED_STATEMENT,LOADED,,2026-11-18T17:05:11Z
1,FM.TUTORIALS.ONLINE_FEDERATED.POSTGRES_SOURCE,SOURCE_DB,LOADED,,2026-11-18T17:05:11Z
2,FM.TUTORIALS.ONLINE_FEDERATED.REDIS_SOURCE,SOURCE_KV,LOADED,,2026-11-18T17:05:11Z
3,FM.TUTORIALS.REALTIME_SEGMENTATION.REDIS_SOURCE,SOURCE_KV,LOADED,,2026-11-18T17:05:12Z


This refresh is about the **source definition**—its URL, timeout, and connection—not the customer values stored behind it.

## Evaluate the starting population

Bind all three customer IDs and return the source values beside the decision:


In [7]:
%%featureql --client serving_client

RETURN
    FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID := BIND_VALUES(ARRAY[1, 2, 3]),
    FM.TUTORIALS.REALTIME_SEGMENTATION.DAYS_SINCE_ORDER,
    FM.TUTORIALS.REALTIME_SEGMENTATION.LIFETIME_VALUE_CENTS,
    FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
;

,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,FM.TUTORIALS.REALTIME_SEGMENTATION.DAYS_SINCE_ORDER,FM.TUTORIALS.REALTIME_SEGMENTATION.LIFETIME_VALUE_CENTS,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
0,1,45,15000,True
1,2,10,8000,False
2,3,60,5000,False


Customer `1` passes both thresholds. Customer `2` is too recent and below $100; customer `3` is old enough but below $100.

## Observe a live Redis update

Now simulate a reverse-ETL update for customer `2`. Its Redis hash changes to 40 days since the last order and 20000 cents in lifetime value.

The next query reads the new values and reevaluates the existing feature:


In [8]:
_result = SERVING_EXECUTORS["redis"].statement("""HSET tutorial:segment:2 days_since_order 40 lifetime_value_cents 20000
""")
if _result.errors:
    raise RuntimeError(_result.errors)

In [9]:
%%featureql --client serving_client

RETURN
    FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID := BIND_VALUE(2),
    FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
;

,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
0,2,True


Customer `2` now returns true. No refresh was needed because the source definition did not change; only the data behind the connection changed.

## Compile the rule for serving

An ad-hoc serving query still needs to compile and plan the FeatureQL expression. `PREPARED_STATEMENT()` compiles `SHOW_PROMO` once and declares `CUSTOMER_ID` as its input:


In [10]:
%%featureql --client serving_client

CREATE OR REPLACE FEATURE FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS AS
PREPARED_STATEMENT(
    FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
    USING FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID
);

,FEATURE_NAME,STATUS,MESSAGE
0,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS,CREATED,Feature created as not exists


Like a new source, a newly persisted prepared statement must be refreshed into the serving process:


In [11]:
%%featureql --client serving_client

REFRESH SERVING;

,FEATURE,KIND,STATUS,MESSAGE,SINCE
0,FM.TUTORIALS.ONLINE_FEDERATED.LAST_ORDER_PRICE_PS,PREPARED_STATEMENT,LOADED,,2026-11-18T17:05:11Z
1,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS,PREPARED_STATEMENT,LOADED,,2026-11-18T17:05:12Z
2,FM.TUTORIALS.ONLINE_FEDERATED.POSTGRES_SOURCE,SOURCE_DB,LOADED,,2026-11-18T17:05:11Z
3,FM.TUTORIALS.ONLINE_FEDERATED.REDIS_SOURCE,SOURCE_KV,LOADED,,2026-11-18T17:05:11Z
4,FM.TUTORIALS.REALTIME_SEGMENTATION.REDIS_SOURCE,SOURCE_KV,LOADED,,2026-11-18T17:05:12Z


Call it first with all three customers, then with customer `2` alone:


In [12]:
import json

_prepared_id = "FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS"
_prepared_calls = [
    '{"input_1": [[1], [2], [3]]}',
    '{"input_1": [[2]]}',
]
for _payload in _prepared_calls:
    _inputs = json.loads(_payload) if isinstance(_payload, str) else _payload
    _result = serving_client.execute_prepared_statement(_prepared_id, _inputs)
    if _result.errors:
        raise RuntimeError(_result.errors)
    display(_result.dataframe)

,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS
0,1,True
1,2,True
2,3,False


,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS
0,2,True


The first response returns true for customers `1` and `2`, and false for `3`. The second response contains only customer `2`. Each payload becomes a separate input table and receives a separate result.

## Full query vs prepared statement

The notebook can compare both call paths. The first loop sends the full FeatureQL query 100 times; the second reuses the prepared plan and changes only its input rows:


In [13]:
import time

_N = 100
_full_query = """
RETURN
    FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,
    FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
FOR
    FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID := BIND_VALUES(ARRAY[1, 2, 3])
;
"""
_prepared_id = "FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS"
_prepared_inputs = {"input_1": [[1], [2], [3]]}

_t0 = time.perf_counter()
for _ in range(_N):
    _result = serving_client.query(_full_query)
    if _result.errors:
        raise RuntimeError(_result.errors)
_full_seconds = time.perf_counter() - _t0

_t0 = time.perf_counter()
for _ in range(_N):
    _result = serving_client.execute_prepared_statement(_prepared_id, _prepared_inputs)
    if _result.errors:
        raise RuntimeError(_result.errors)
_prepared_seconds = time.perf_counter() - _t0

print(f"Full FeatureQL query × {_N}: {_full_seconds:.3f}s ({_full_seconds / _N * 1000:.2f} ms/call)")
print(f"Prepared statement × {_N}: {_prepared_seconds:.3f}s ({_prepared_seconds / _N * 1000:.2f} ms/call)")
print(f"Speedup: {_full_seconds / _prepared_seconds:.1f}×")

Full FeatureQL query × 100: 6.280s (62.80 ms/call)
Prepared statement × 100: 0.123s (1.23 ms/call)
Speedup: 51.0×


Treat the printed timings as a local observation, not a universal benchmark. Network latency, warm-up, and the serving environment all affect the ratio. The architectural difference is stable: the prepared path avoids recompiling the same feature graph on every request.

## What's next

You have separated source installation from live source data, then exposed the same promotion rule through an ad-hoc query and a prepared statement.

Continue with [Federated serving](https://featuremesh.com/docs/tutorials/serving/federation) to follow one feature graph across PostgreSQL and Redis. Use the [Redis source](https://featuremesh.com/docs/featuremesh/realtime_serving/external_kv) and [prepared statement](https://featuremesh.com/docs/featuremesh/realtime_serving/prepared_statements) pages when you need complete syntax and payload options.


## Clean up

Drop tutorial features and fixture data so you can re-run the notebook from a clean state. Clean-up is off by default, so that running all cells keeps what the tutorial built: set `RUN_CLEANUP = True` below, then run the clean-up cells.


In [14]:
# Clean-up is off so that "Run All" keeps what the tutorial built.
# Set it to True and run this cell and the cells below to drop the tutorial features and data.
RUN_CLEANUP = False

In [15]:
_cleanup_query = """DROP FEATURES IF EXISTS IN FM.TUTORIALS.REALTIME_SEGMENTATION UP TO LEVEL 9;
"""
if RUN_CLEANUP:
    get_ipython().run_cell_magic("featureql", "--client serving_client --hide-dataframe", _cleanup_query)

In [16]:
_cleanup_command = """DEL tutorial:segment:1 tutorial:segment:2 tutorial:segment:3
"""
if RUN_CLEANUP:
    _result = SERVING_EXECUTORS["redis"].statement(_cleanup_command)
    if _result.errors:
        raise RuntimeError(_result.errors)

---

Source tutorial: [/docs/tutorials/serving/realtime](https://featuremesh.com/docs/tutorials/serving/realtime)
